# Multi-Step Sales Research Workflow (LangGraph)

This notebook implements a LangGraph workflow for:
1. Company research
2. Lead summarization
3. Outreach draft generation
4. QA review


## Why This Workflow

Sales research is usually fragmented across tools.
A graph workflow lets each step write structured outputs into shared state,
so downstream steps can reason over upstream context consistently.


## State Passing and Node Design

### Node Design Principles
- Each node has one clear responsibility.
- Each node reads only the fields it needs.
- Each node writes explicit fields for later nodes.

### State Passing
LangGraph passes one shared typed state object through the graph.

```text
START
  -> company_research      writes: company_profile, company_pain_points
  -> lead_summarization    writes: target_lead_summary, value_hypothesis
  -> outreach_draft        writes: outreach_email
  -> qa_review             writes: qa_result, qa_notes, final_email
  -> END
```

Because each node writes structured fields, the next node receives richer context.


## Step 1 - Install and Import

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph'])
print('langgraph installed')


langgraph installed


In [2]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

print('Imports ready')


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports ready


## Step 2 - Define State Schema

In [3]:
class SalesResearchState(TypedDict):
    target_company: str
    product_name: str
    product_value: str
    company_profile: str
    company_pain_points: str
    target_lead_summary: str
    value_hypothesis: str
    outreach_email: str
    qa_result: str
    qa_notes: str
    final_email: str

print('State schema ready')


State schema ready


## Step 3 - Simulated Research Sources

In [4]:
COMPANY_DB = {
    'northstar manufacturing': {
        'industry': 'Industrial manufacturing',
        'size': '4200 employees',
        'region': 'US + EU',
        'recent_initiative': 'Launched multi-site ERP modernization in 2025',
        'pain_points': 'slow reporting cycles, disconnected plant data, delayed executive dashboards',
        'target_lead': 'VP Operations, Priya Shah',
        'lead_context': 'Owns plant efficiency targets and cross-site reporting standardization',
    },
    'bluewave health': {
        'industry': 'Health services',
        'size': '1600 employees',
        'region': 'US',
        'recent_initiative': 'Scaled virtual care platform after Series C expansion',
        'pain_points': 'compliance-heavy data pipelines, slow analytics requests, PHI governance friction',
        'target_lead': 'Director of Data Platform, Marcus Lee',
        'lead_context': 'Leads data platform reliability and analytics delivery for clinical teams',
    },
}

print('Companies available:', list(COMPANY_DB.keys()))


Companies available: ['northstar manufacturing', 'bluewave health']


## Step 4 - Node 1: Company Research

In [5]:
def company_research(state: SalesResearchState) -> dict:
    key = state['target_company'].strip().lower()
    if key not in COMPANY_DB:
        profile = 'Unknown company profile. Manual research required.'
        pains = 'Unknown pain points.'
    else:
        rec = COMPANY_DB[key]
        profile = (
            'Company=' + state['target_company'] + '; '
            'Industry=' + rec['industry'] + '; '
            'Size=' + rec['size'] + '; '
            'Region=' + rec['region'] + '; '
            'Initiative=' + rec['recent_initiative']
        )
        pains = rec['pain_points']

    print('company_research completed for', state['target_company'])
    return {'company_profile': profile, 'company_pain_points': pains}


## Step 5 - Node 2: Lead Summarization

In [6]:
def lead_summarization(state: SalesResearchState) -> dict:
    key = state['target_company'].strip().lower()
    if key in COMPANY_DB:
        rec = COMPANY_DB[key]
        lead_summary = rec['target_lead'] + ' - ' + rec['lead_context']
    else:
        lead_summary = 'Default target: Operations or Data leader (manual validation needed).'

    hypothesis = (
        'Given pain points (' + state['company_pain_points'] + '), propose ' + state['product_name'] +
        ' as a fast path to improve reporting reliability and decision speed.'
    )

    print('lead_summarization completed')
    return {'target_lead_summary': lead_summary, 'value_hypothesis': hypothesis}


## Step 6 - Node 3: Outreach Draft Generation

In [7]:
def outreach_draft(state: SalesResearchState) -> dict:
    subject = 'Subject: Quick idea for ' + state['target_company']
    body = (
        'Hi,\n\n'
        'I noticed ' + state['target_company'] + ' is working on ' + state['company_profile'].split('Initiative=')[-1] + '. '
        'Given current friction around ' + state['company_pain_points'] + ', '
        'teams similar to yours use ' + state['product_name'] + ' to ' + state['product_value'] + '. '
        'Potential owner on your side appears to be ' + state['target_lead_summary'] + '.\n\n'
        'If useful, I can share a concise 15-minute walkthrough focused on your rollout priorities.\n\n'
        'Best regards,\nSales Team'
    )
    draft = subject + '\n\n' + body
    print('outreach_draft completed')
    return {'outreach_email': draft}


## Step 7 - Node 4: QA Review

In [8]:
def qa_review(state: SalesResearchState) -> dict:
    email = state['outreach_email']
    word_count = len(email.split())

    issues = []
    if word_count > 170:
        issues.append('Email too long for first-touch outreach.')
    if 'guarantee' in email.lower() or 'act now' in email.lower():
        issues.append('Contains aggressive or risky language.')
    if state['target_company'].lower() not in email.lower():
        issues.append('Missing company personalization.')

    if issues:
        result = 'FAIL'
        notes = ' | '.join(issues)
        final_email = ''
    else:
        result = 'PASS'
        notes = 'Meets quality guardrails: concise, personalized, low-pressure CTA.'
        final_email = email

    print('qa_review completed with', result)
    return {'qa_result': result, 'qa_notes': notes, 'final_email': final_email}


## Step 8 - Build LangGraph Pipeline

In [9]:
builder = StateGraph(SalesResearchState)
builder.add_node('company_research', company_research)
builder.add_node('lead_summarization', lead_summarization)
builder.add_node('outreach_draft', outreach_draft)
builder.add_node('qa_review', qa_review)

builder.add_edge(START, 'company_research')
builder.add_edge('company_research', 'lead_summarization')
builder.add_edge('lead_summarization', 'outreach_draft')
builder.add_edge('outreach_draft', 'qa_review')
builder.add_edge('qa_review', END)

app = builder.compile()
print('Graph compiled')


Graph compiled


## Step 9 - Run Workflow for Two Target Companies

In [10]:
inputs = [
    {
        'target_company': 'NorthStar Manufacturing',
        'product_name': 'PipelinePilot',
        'product_value': 'automate cross-system data delivery and shorten reporting turnaround',
        'company_profile': '',
        'company_pain_points': '',
        'target_lead_summary': '',
        'value_hypothesis': '',
        'outreach_email': '',
        'qa_result': '',
        'qa_notes': '',
        'final_email': '',
    },
    {
        'target_company': 'BlueWave Health',
        'product_name': 'PipelinePilot',
        'product_value': 'deliver compliant analytics workflows with lower manual overhead',
        'company_profile': '',
        'company_pain_points': '',
        'target_lead_summary': '',
        'value_hypothesis': '',
        'outreach_email': '',
        'qa_result': '',
        'qa_notes': '',
        'final_email': '',
    },
]

results = []
for item in inputs:
    results.append(app.invoke(item))

print('Runs completed:', len(results))


company_research completed for NorthStar Manufacturing
lead_summarization completed
outreach_draft completed
qa_review completed with PASS
company_research completed for BlueWave Health
lead_summarization completed
outreach_draft completed
qa_review completed with PASS
Runs completed: 2


## Step 10 - Inspect Outputs

In [11]:
for r in results:
    print('=' * 80)
    print('Company:', r['target_company'])
    print('Lead summary:', r['target_lead_summary'])
    print('Value hypothesis:', r['value_hypothesis'])
    print('QA result:', r['qa_result'])
    print('QA notes:', r['qa_notes'])
    print('Outreach draft preview:', r['outreach_email'][:220])


Company: NorthStar Manufacturing
Lead summary: VP Operations, Priya Shah - Owns plant efficiency targets and cross-site reporting standardization
Value hypothesis: Given pain points (slow reporting cycles, disconnected plant data, delayed executive dashboards), propose PipelinePilot as a fast path to improve reporting reliability and decision speed.
QA result: PASS
QA notes: Meets quality guardrails: concise, personalized, low-pressure CTA.
Outreach draft preview: Subject: Quick idea for NorthStar Manufacturing

Hi,

I noticed NorthStar Manufacturing is working on Launched multi-site ERP modernization in 2025. Given current friction around slow reporting cycles, disconnected plant
Company: BlueWave Health
Lead summary: Director of Data Platform, Marcus Lee - Leads data platform reliability and analytics delivery for clinical teams
Value hypothesis: Given pain points (compliance-heavy data pipelines, slow analytics requests, PHI governance friction), propose PipelinePilot as a fast path

## Step 11 - Final Approved Outreach Drafts

In [12]:
approved = [r for r in results if r['qa_result'] == 'PASS']
print('Approved drafts:', len(approved))

for r in approved:
    print('\n' + '=' * 80)
    print('FINAL EMAIL for', r['target_company'])
    print('=' * 80)
    print(r['final_email'])


Approved drafts: 2

FINAL EMAIL for NorthStar Manufacturing
Subject: Quick idea for NorthStar Manufacturing

Hi,

I noticed NorthStar Manufacturing is working on Launched multi-site ERP modernization in 2025. Given current friction around slow reporting cycles, disconnected plant data, delayed executive dashboards, teams similar to yours use PipelinePilot to automate cross-system data delivery and shorten reporting turnaround. Potential owner on your side appears to be VP Operations, Priya Shah - Owns plant efficiency targets and cross-site reporting standardization.

If useful, I can share a concise 15-minute walkthrough focused on your rollout priorities.

Best regards,
Sales Team

FINAL EMAIL for BlueWave Health
Subject: Quick idea for BlueWave Health

Hi,

I noticed BlueWave Health is working on Scaled virtual care platform after Series C expansion. Given current friction around compliance-heavy data pipelines, slow analytics requests, PHI governance friction, teams similar to your

## Recap: Node Responsibilities and State Flow

1. `company_research` populates business context.
2. `lead_summarization` converts context into a target contact narrative and value hypothesis.
3. `outreach_draft` uses all prior fields to generate personalized outreach.
4. `qa_review` gates output quality and either approves or blocks final email.

This structure keeps each node testable and keeps state passing explicit.
